# watch

> recurring acquisition and reminders

In [ ]:
#| default_exp watch

In [ ]:
#| hide
from nbdev.showdoc import *

A watch is what turns the vault from an archive into something that stays current. Actions reuse the
ordinary acquisition methods, so anything you can file once you can file on a schedule.

In [ ]:
#| export
import json, re, time, uuid
from fastcore.all import AttrDict, L, patch, first
from vishalakshi.core import Vault

In [ ]:
#| export
ACTIONS = ('url', 'web', 'harvest', 'arxiv', 'youtube', 'crawl', 'remind')

_UNITS = dict(s=1, m=60, h=3600, d=86400, w=604800)
_EVERY = re.compile(r'^\s*(\d+(?:\.\d+)?)\s*([smhdw])\w*\s*$', re.I)

def parse_every(every) -> float:
    "Seconds from `'30m'`, `'6h'`, `'2 days'`, or a number of seconds."
    if isinstance(every, (int, float)): return float(every)
    m = _EVERY.match(str(every or ''))
    if not m: raise ValueError(f"can't read an interval from {every!r} — try '30m', '6h', '1d', '1w'")
    return float(m.group(1)) * _UNITS[m.group(2).lower()]

def fmt_every(secs:float) -> str:
    'Seconds back to the shortest readable interval.'
    for u, n in (('w', 604800), ('d', 86400), ('h', 3600), ('m', 60)):
        if secs >= n and secs % n == 0: return f'{int(secs//n)}{u}'
    return f'{int(secs)}s'

In [ ]:
#| export
@patch
def _watches(self:Vault):
    'The watches table, created on first use.'
    t = self.db.t.watches
    t.create(id=str, action=str, target=str, params=str, every=float, note=str, enabled=int,
             last_run=float, last_status=str, next_run=float, runs=int, created_at=float,
             pk='id', if_not_exists=True)
    return t

@patch
def watch(self:Vault,
          target:str,         # URL, query, arXiv id, or the text of a reminder
          action:str='url',   # one of ACTIONS — what to do when it fires
          every='1d',         # interval: '30m', '6h', '1d', '1w', or seconds
          note:str=None,      # why you are watching; written into the vault when it fires
          start:float=None,   # first run time (epoch); defaults to now
          **params            # forwarded to the action (n=, pattern=, pages=, sel=, ...)
) -> dict:
    """Register a recurring job: re-read a page, re-run a search, re-harvest an API, or remind you.

    A watch is what turns the vault from an archive into something that stays current. `action`
    reuses the ordinary acquisition methods, so anything you can put in the vault once you can put
    in it on a schedule; `action='remind'` writes a note instead of fetching, which is the
    recurring-reminder case with no network involved."""
    assert action in ACTIONS, f'action must be one of {ACTIONS}'
    secs = parse_every(every)
    now = time.time()
    row = dict(id=uuid.uuid4().hex[:12], action=action, target=target,
               params=json.dumps(params or {}), every=secs, note=note or '', enabled=1,
               last_run=None, last_status=None, next_run=start or now, runs=0, created_at=now)
    self._watches().insert(row, replace=True)
    return dict(row, params=params or {}, every=fmt_every(secs))

@patch
def watches(self:Vault, enabled_only:bool=False) -> list:
    'Every registered watch, soonest first.'
    rows = list(self._watches()())
    if enabled_only: rows = [r for r in rows if r['enabled']]
    for r in rows:
        r['params'] = json.loads(r['params'] or '{}')
        r['every'] = fmt_every(r['every'])
    return sorted(rows, key=lambda r: r['next_run'] or 0)

@patch
def due(self:Vault, at:float=None) -> list:
    'Watches whose next run has arrived.'
    now = at or time.time()
    return [w for w in self.watches(enabled_only=True) if (w['next_run'] or 0) <= now]

@patch
def unwatch(self:Vault, watch_id:str):
    'Delete a watch. The documents it already filed stay in the vault.'
    self._watches().delete_where(where=f'id={watch_id!r}')

@patch
def pause(self:Vault, watch_id:str, enabled:bool=False):
    'Disable (or re-enable) a watch without losing it.'
    self._watches().update(dict(id=watch_id, enabled=int(enabled)))

In [ ]:
#| export
@patch
def run_watch(self:Vault, w:dict) -> dict:
    """Fire one watch and record the outcome.

    A failure is recorded on the row and returned, never raised: one dead URL must not stop a
    polling loop from servicing every other watch."""
    a, t, p = w['action'], w['target'], (w.get('params') or {})
    started = time.time()
    try:
        if   a == 'remind':  res = self.note(t, title=w.get('note') or None, tags=['reminder'])
        elif a == 'url':     res = self.url(t, **p)
        elif a == 'web':     res = self.web(t, **p)
        elif a == 'harvest': res = self.harvest(t, **p)
        elif a == 'arxiv':   res = self.arxiv(t, **p)
        elif a == 'youtube': res = self.youtube(t, **p)
        elif a == 'crawl':   res = self.crawl(t, **p)
        else: raise ValueError(f'unknown action {a!r}')
        status = 'skipped' if isinstance(res, dict) and res.get('skipped') else 'ok'
    except Exception as e:
        res, status = dict(error=f'{type(e).__name__}: {str(e)[:200]}'), 'error'
    now = time.time()
    self._watches().update(dict(id=w['id'], last_run=now, last_status=status,
                                next_run=now + w_every(w), runs=(w.get('runs') or 0) + 1))
    return dict(watch_id=w['id'], action=a, target=t, status=status,
                took=round(now - started, 2), result=res)

def w_every(w) -> float:
    'A watch row\'s interval in seconds, whether it is stored raw or formatted.'
    e = w.get('every')
    return parse_every(e) if not isinstance(e, (int, float)) else float(e)

@patch
def poll(self:Vault, at:float=None, limit:int=None, connect:bool=True) -> dict:
    """Run every watch that is due. This is the tick a scheduler, a cron or a frontend calls.

    Rebuilds the entity graph once at the end rather than per watch, because `connect()` reads the
    whole store and a poll that fired five watches would otherwise pay for it five times."""
    due = self.due(at)[:limit] if limit else self.due(at)
    ran = [self.run_watch(w) for w in due]
    if connect and any(r['status'] == 'ok' for r in ran): self.connect()
    return dict(checked=len(self.watches(enabled_only=True)), ran=len(ran), results=ran,
                next_due=first(sorted((w['next_run'] or 0) for w in self.watches(enabled_only=True))))